Requested Plots to Make:

1. Plot inbound and outbound path traces
    i. Asked to see outbound and inbound path traces
    ii. Asked to look at supposed two different strategirs for searching (inbound paths I believe)
    - Should be done for:
        - individual animals
        - multiple animals
        - different sessions

    My interpretation:
        i. For each mouse, plot outbound paths in one column, inbound paths in next column. Each row should be one session (or day, should use columns of dataframe/dataarray to set this as input parameter to function, ordered from earliest to latest). Trials should use a colourmap, normalised so that it covers 0.2 to 1.0, with earlier trials for each session being lighter. 
        ii. Second plot should be like the first, but without outbound paths. Should have input parameter like `groups = {'g1': {'mouse1', 'mouse2'}, 'g2': {'mouse3'}}`.

        Both plots should use `movement` python package functions to plot these path traces using DLC pose data. Should take centroid of datapoints passed as input list to function, i.e. `centroid_points = {'nose', 'torso', 'rightarm', 'rightleg'}`, but by default should just be all the pose points in the DLC data (read the data and actually hardcode this). this is more in case it needs to change. 
        Both plots should also overlay a static frame at the start, perhaps the first frame of the video recording. 
        Both plots should have fixed axes so that the entire space is visible. 

## Implementation: `plot_path_grid`

One function (`data_conduit.qc.plot_path_grid`) draws both figures. Rows and columns are each any trials-table column (`session`, `day`, `group`, `mouseID`) or the special `'segment'` (the outbound|inbound pair). Grouping just adds a `group` column via `groups={...}`, so it's another axis you can toggle onto rows or columns.

- Colouring defaults to per mouse, but grouped calls switch to per group unless you pass `color_by='mouseID'`; trials ramp **0.2 → 1.0** in trial order (earlier lighter).
- Per-session first `.avi` frame as a static background; fixed pixel axes.
- Centroid over `centroid_points`, default all keypoints: `('nose', 'lear', 'rear', 'body', 'tailbase')`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, data_conduit
from data_conduit.refactor_qc import qc_datastructure, plot_path_grid

# Anchor to the repo root so ROOT resolves regardless of the notebook's working dir.
REPO = pathlib.Path(data_conduit.__file__).resolve().parents[2]
ROOT = REPO / 'datasets/firstdata/BonsaiFiles'

ds = qc_datastructure(ROOT, depth=2, level_names=('mouseID', 'day'),
                      streams=('dlc',), l1_selector='Day 11')
result = ds.load()

# Keep only trial rows from sessions that are actually present in the DLC pose stream.
# The full result may include trial-only sessions because trials and DLC are loaded as
# separate streams.
dlc_sessions = set(result['dlc:position'].coords['session'].values)
filtered_results = result.copy()
filtered_results['trials'] = result['trials'][
    result['trials']['session'].isin(dlc_sessions)
].copy()

sorted(set(result['trials']['mouseID']))

In [ ]:
print(f'Result keys: {result.keys()}')

display(result['trials'])
display(result['dlc:position'].to_dataframe())

print(f'Number of sessions: {result["trials"]["session"].nunique()}')
print(f'Number of sessions with DLC: {filtered_results["trials"]["session"].nunique()}')

display(filtered_results['trials'])

### Plot 1 — outbound vs inbound, per mouse

Rows = sessions (earliest → latest), columns = outbound | inbound.

In [ ]:
fig, axes = plot_path_grid(
    filtered_results, ROOT,
    mice=['FbR_M01569522'],
    row_by='session', col_by='segment',        # columns = outbound | inbound
    # centroid_points=('nose', 'body'),          # optional: subset for the centroid
)

### Plot 2 — all mice as columns, outbound + inbound each

Nest axes by passing a list: `col_by=['mouseID', 'segment']` gives one column block per mouse, each split into adjacent outbound | inbound. Rows are `day` (a day spans mice; `session` rows would be sparse since each session belongs to one mouse). Each mouse keeps its own colourmap, and the trial ramp spans the whole day.

To compare **groups** instead, pass `groups={'g1': {...}, 'g2': {...}}` and use `col_by=['group', 'segment']`; grouped plots colour by group by default.

In [ ]:
fig, axes = plot_path_grid(
    filtered_results, ROOT,
    row_by='day', col_by=['mouseID', 'segment'],   # columns = mouse x [outbound, inbound]
)

### Plot 3 — outbound & inbound split by outcome, per group

`outcome` (Success / Miss / Failure) is just another axis. Rows = outcome, columns = group × segment. Cells with no (valid) trials show the bare frame.

Traces are drawn with `movement`'s `plot_centroid_trajectory`. Each subpanel carries its active colour label (`group  n=…` here, or `mouseID  n=…` with `color_by='mouseID'`) and a slim trial colourbar (`colorbar=False` to drop them).

In [ ]:
fig, axes = plot_path_grid(
    filtered_results, ROOT,
    groups={'g1': {'FbR_M01569522', 'FLR_M01569521'}},
    color_by='group',                  # explicit for clarity; this is now the default with groups=
    row_by='outcome',                    # Success / Miss / Failure
    col_by=['group', 'segment'],         # columns = group x [outbound, inbound]
)

### Plot 4 — one session, one outbound|inbound pair per trial

Pick one session from `available_sessions`, then use `trial_index` for rows and `segment` for columns so each trial gets an outbound panel beside its inbound panel.

In [ ]:
available_sessions = sorted(filtered_results['trials']['session'].unique())
session_id = available_sessions[0]  # change this to the session you want
session_results = filtered_results.copy()
session_results['trials'] = filtered_results['trials'].loc[
    filtered_results['trials']['session'].eq(session_id)
].copy()

fig, axes = plot_path_grid(
    session_results, ROOT,
    row_by='trial_index',               # one row per trial in this session
    col_by='segment',                   # outbound | inbound
    colorbar=False,                     # cleaner when there are many rows
    fontsize=10,
    figsize_per_cell=(5.6, 3.8),
)
fig.suptitle(f'Session {session_id}: outbound | inbound per trial', y=1.01)

In [ ]:
from data_conduit.refactor_qc.slicing import slice_pose_for_trial

trials = filtered_results["trials"]
pose = filtered_results["dlc:position"]

sum(
    slice_pose_for_trial(pose, row, segment="inbound").sizes.get("Time", 0) > 0
    for _, row in trials.iterrows()
)